# VehiGuard - train the Indian plate detector (task 2.4)

Run this notebook on **Google Colab with a free GPU**:
*Runtime -> Change runtime type -> T4 GPU*, then *Runtime -> Run all*.

It downloads your labelled dataset from Roboflow (task 2.3, see `docs/plate_dataset_guide.md`),
fine-tunes YOLOv8 starting from the Nepal baseline (D07), reports mAP, precision and recall,
and downloads `best.pt` for you.

In [ ]:
# Same Ultralytics version as the laptop, so the model loads there without problems
!pip install -q ultralytics==8.4.174 roboflow huggingface_hub
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 1. Download your dataset from Roboflow

In Roboflow: open your project -> **Versions** -> your version -> **Download Dataset** ->
format **YOLOv8** -> *show download code*. Copy the workspace, project and version
into the cell below. The API key is typed in when asked (it is not saved in the notebook).

In [ ]:
from getpass import getpass
from roboflow import Roboflow

WORKSPACE = "your-workspace"   # <- change
PROJECT = "indian-plates"      # <- change
VERSION = 1                    # <- change

rf = Roboflow(api_key=getpass("Roboflow API key: "))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print("Dataset in", dataset.location)
!cat {dataset.location}/data.yaml

## 2. Fine-tune

Starts from the Nepal plate model (it already knows what plates look like), so it needs
fewer images and epochs. Training stops early when it stops improving.

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

start = hf_hub_download("krishnamishra8848/Nepal-Vehicle-License-Plate-Detection", "last.pt")
model = YOLO(start)
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60, patience=15, imgsz=640, batch=16,
    project="runs", name="plate", exist_ok=True,
)

## 3. Results on the test split (for your report)

In [ ]:
best = YOLO("runs/plate/weights/best.pt")
metrics = best.val(data=f"{dataset.location}/data.yaml", split="test")
print(f"mAP50     {metrics.box.map50:.3f}")
print(f"mAP50-95  {metrics.box.map:.3f}")
print(f"Precision {metrics.box.mp:.3f}")
print(f"Recall    {metrics.box.mr:.3f}")

In [ ]:
from IPython.display import Image, display
for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    try:
        display(Image(filename=f"runs/plate/{name}", width=900))
    except FileNotFoundError:
        pass

## 4. Download the model

In [ ]:
from google.colab import files
files.download("runs/plate/weights/best.pt")

## 5. Use it on the laptop

1. Rename the downloaded file to `plate_detector.pt` and put it in the `models/` folder of the project.
2. Compare it with the Nepal baseline on your held-out test photos:
   ```
   python scripts/evaluate.py data/plate_test --labels data/plate_test/labels --name baseline
   python scripts/evaluate.py data/plate_test --labels data/plate_test/labels --weights models/plate_detector.pt --name finetuned
   ```
3. If it is better, set `plate_model: models/plate_detector.pt` in `config/settings.yaml`.
4. Record the model in `docs/datasets.md` (name, date, dataset version, metrics).